In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()

spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
sc = spark.sparkContext

In [ ]:
orders_base_rdd = sc.textFile("data/retail_db_orders_part-00000")
customers_base_rdd = sc.textFile("data/retail_db_customers_part-00000")
order_items_base_rdd = sc.textFile("data/retail_db_order_items_part-00000")


# The data is stored in HDFS as shown below:
[user@g02 ~]$ hdfs dfs -ls -h ./data
Found 5 items
drwxr-xr-x   - user supergroup          0 2026-03-03 09:28 data/datasets
drwxr-xr-x   - user supergroup          0 2026-03-12 13:07 data/external_table
-rw-r--r--   3 user supergroup    931.4 K 2026-02-25 09:42 data/retail_db_customers_part-00000
-rw-r--r--   3 user supergroup      5.2 M 2026-02-22 10:10 data/retail_db_order_items_part-00000
-rw-r--r--   3 user supergroup      2.9 M 2026-02-21 09:43 data/retail_db_orders_part-00000
[user@g02 ~]$ 

## 2. Top 10 product IDs with most quantity sold

In [4]:
# order_item_product_id, order_item_quantity
order_items_columns = order_items_base_rdd.map(lambda x: (x.split(",")[2], int(x.split(",")[3])))

In [5]:
order_items_columns.take(10)

[('957', 1),
 ('1073', 1),
 ('502', 5),
 ('403', 1),
 ('897', 2),
 ('365', 5),
 ('502', 3),
 ('1014', 4),
 ('957', 1),
 ('365', 5)]

In [6]:
order_items_columns_quantity = order_items_columns.reduceByKey(lambda x,y: x+y).sortBy(lambda x: x[1], False)

In [ ]:
order_items_columns_quantity.take(10)

[('365', 73698),
 ('502', 62956),
 ('1014', 57803),
 ('191', 36680),
 ('627', 31735),
 ('403', 22246),
 ('1004', 17325),
 ('1073', 15500),
 ('957', 13729),
 ('977', 998)]